## Computational Cost of the ML-based B-Point Detection Approach

This notebook estimates the computational cost of the ML-based B-point regression approach (MinMaxScaler + RandomForestRegressor trained on the outputs of the 12 classical B-point algorithms plus the estimated RR-interval, cf. `Analyse_Model_Parameters.ipynb`).

We report:
- **Training time** of the regressor
- **Batch inference time** (throughput)
- **Single-sample inference latency** (real-time / per-heartbeat use case)
- **Model size / complexity** (serialized size, number of trees and nodes)
- **Feature extraction time** of the underlying 12 classical B-point algorithms (plus heartbeat segmentation and C-point extraction) that produce the regressor's input feature vector
- **End-to-end latency** of the full ML pipeline (feature extraction + regression) for a single heartbeat

Feature extraction timings are measured on the small example ECG/ICG recording bundled with `pepbench` (`pepbench.example_data.get_example_dataset`), since running the classical algorithms requires the raw signal (which is not part of the precomputed feature CSVs used for the regressor).

### Setup and Helper Functions

#### Imports

In [10]:
import pickle
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import biopsykit as bp
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

from pepbench.utils._timing import measure_time

from pepbench.example_data import get_example_dataset
from pepbench.algorithms.heartbeat_segmentation import HeartbeatSegmentationNeurokit
from biopsykit.signals.icg.event_extraction import CPointExtractionScipyFindPeaks
from pepbench.algorithms.icg import (
    BPointExtractionArbol2017IsoelectricCrossings,
    BPointExtractionArbol2017SecondDerivative,
    BPointExtractionArbol2017ThirdDerivative,
    BPointExtractionDebski1993SecondDerivative,
    BPointExtractionDrost2022,
    BPointExtractionForouzanfar2018,
    BPointExtractionLozano2007LinearRegression,
    BPointExtractionLozano2007QuadraticRegression,
    BPointExtractionMiljkovic2022,
    BPointExtractionPale2021,
    BPointExtractionSherwood1990,
    BPointExtractionStern1985,
)


#### Datapaths

In [11]:
data_path = Path("../../results")
model_path = Path("../../models")
rater = "rater_01"


### Feature extraction time (classical B-point algorithms)

The ML regressor's input feature vector consists of the B-point estimates produced by the 12 classical algorithms (the same set used in the benchmarking challenge notebooks) plus the estimated RR-interval, which is a byproduct of heartbeat segmentation. We measure the time to compute these features from the raw ECG/ICG signal on the example dataset bundled with `pepbench`, so that the reported cost of the ML approach also covers this feature-extraction step and not just the regressor itself.

In [16]:
example_dataset = get_example_dataset()
example_subset = example_dataset.get_subset(participant="VP_001")

fs_ecg = example_subset.sampling_rate_ecg
fs_icg = example_subset.sampling_rate_icg
ecg_data = example_subset.ecg
icg_data = example_subset.icg

n_repeats_extraction = 20


#### Heartbeat segmentation and C-point extraction

Both steps are shared across all B-point algorithms and only need to be computed once per recording.

In [13]:
heartbeat_times = []
for _ in range(n_repeats_extraction):
    heartbeat_segmentation_algo = HeartbeatSegmentationNeurokit()
    with measure_time() as timing_results:
        heartbeat_segmentation_algo.extract(ecg=ecg_data, sampling_rate_hz=fs_ecg)
    heartbeat_times.append(timing_results["runtime_s"])

heartbeats = heartbeat_segmentation_algo.heartbeat_list_
n_heartbeats = heartbeats.shape[0]
heartbeat_times = np.array(heartbeat_times)

c_point_times = []
for _ in range(n_repeats_extraction):
    c_point_algo = CPointExtractionScipyFindPeaks()
    with measure_time() as timing_results:
        c_point_algo.extract(icg=icg_data, heartbeats=heartbeats, sampling_rate_hz=fs_icg)
    c_point_times.append(timing_results["runtime_s"])

c_points = c_point_algo.points_
c_point_times = np.array(c_point_times)

print(f"Recording length: {n_heartbeats} heartbeats")
print(f"Heartbeat segmentation: {heartbeat_times.mean() * 1000:.2f} ms ± {heartbeat_times.std() * 1000:.2f} ms ({heartbeat_times.mean() / n_heartbeats * 1000:.4f} ms/heartbeat)")
print(f"C-point extraction:     {c_point_times.mean() * 1000:.2f} ms ± {c_point_times.std() * 1000:.2f} ms ({c_point_times.mean() / n_heartbeats * 1000:.4f} ms/heartbeat)")


Recording length: 82 heartbeats
Heartbeat segmentation: 5.56 ms ± 1.06 ms (0.0678 ms/heartbeat)
C-point extraction:     11.06 ms ± 0.51 ms (0.1349 ms/heartbeat)


#### B-point extraction (all 12 classical algorithms)

Each algorithm produces one of the feature columns used by the ML regressor. We time each algorithm separately over the whole recording and normalize by the number of heartbeats.

In [14]:
b_point_algo_dict = {
    "stern1985": BPointExtractionStern1985(),
    "sherwood1990": BPointExtractionSherwood1990(),
    "debski1993-second-derivative": BPointExtractionDebski1993SecondDerivative(),
    "lozano2007-linear-regression": BPointExtractionLozano2007LinearRegression(),
    "lozano2007-quadratic-regression": BPointExtractionLozano2007QuadraticRegression(),
    "arbol2017-isoelectric-crossings": BPointExtractionArbol2017IsoelectricCrossings(),
    "arbol2017-second-derivative": BPointExtractionArbol2017SecondDerivative(),
    "arbol2017-third-derivative": BPointExtractionArbol2017ThirdDerivative(),
    "forouzanfar2018": BPointExtractionForouzanfar2018(),
    "drost2022": BPointExtractionDrost2022(),
    "pale2021": BPointExtractionPale2021(),
    "miljkovic2022": BPointExtractionMiljkovic2022(),
}

b_point_extraction_times = {}
for algo_name, algo in b_point_algo_dict.items():
    algo_times = []
    for _ in range(n_repeats_extraction):
        algo_instance = algo.clone()
        with measure_time() as timing_results:
            algo_instance.extract(icg=icg_data, heartbeats=heartbeats, c_points=c_points, sampling_rate_hz=fs_icg)
        algo_times.append(timing_results["runtime_s"])
    b_point_extraction_times[algo_name] = np.array(algo_times)

b_point_extraction_summary = pd.DataFrame({
    "total_time_ms_mean": {k: v.mean() * 1000 for k, v in b_point_extraction_times.items()},
    "total_time_ms_std": {k: v.std() * 1000 for k, v in b_point_extraction_times.items()},
    "time_per_heartbeat_ms_mean": {k: v.mean() / n_heartbeats * 1000 for k, v in b_point_extraction_times.items()},
})
b_point_extraction_summary.sort_values("time_per_heartbeat_ms_mean")


/Users/abelluc/Code/pepbench/.venv/lib/python3.10/site-packages/biopsykit/signals/icg/event_extraction/_b_point_forouzanfar2018.py:222: UserWarning: Either 'r_peak' or 'c_point' contains NaN at positions [55]! The B-points were set to NaN.
  warnings.warn(missing_str)
/Users/abelluc/Code/pepbench/.venv/lib/python3.10/site-packages/biopsykit/signals/icg/event_extraction/_b_point_forouzanfar2018.py:222: UserWarning: Either 'r_peak' or 'c_point' contains NaN at positions [55]! The B-points were set to NaN.
  warnings.warn(missing_str)
/Users/abelluc/Code/pepbench/.venv/lib/python3.10/site-packages/biopsykit/signals/icg/event_extraction/_b_point_forouzanfar2018.py:222: UserWarning: Either 'r_peak' or 'c_point' contains NaN at positions [55]! The B-points were set to NaN.
  warnings.warn(missing_str)
/Users/abelluc/Code/pepbench/.venv/lib/python3.10/site-packages/biopsykit/signals/icg/event_extraction/_b_point_forouzanfar2018.py:222: UserWarning: Either 'r_peak' or 'c_point' contains NaN at

,total_time_ms_mean,total_time_ms_std,time_per_heartbeat_ms_mean
arbol2017-second-derivative,5.079665,0.315100,0.061947
arbol2017-third-derivative,5.272892,0.365547,0.064304
sherwood1990,5.316483,0.403388,0.064835
debski1993-second-derivative,5.739710,0.281631,0.069996
stern1985,6.298235,0.984817,0.076808
pale2021,14.201006,0.522940,0.173183
arbol2017-isoelectric-crossings,14.398365,0.600420,0.175590
drost2022,24.331952,0.634546,0.296731
miljkovic2022,24.737312,0.685950,0.301675
lozano2007-quadratic-regression,36.495908,1.028500,0.445072


#### Total feature extraction cost per heartbeat

In [15]:
heartbeat_seg_per_heartbeat_ms = heartbeat_times.mean() / n_heartbeats * 1000
c_point_per_heartbeat_ms = c_point_times.mean() / n_heartbeats * 1000
b_point_algos_per_heartbeat_ms = b_point_extraction_summary["time_per_heartbeat_ms_mean"].sum()

feature_extraction_time_per_heartbeat_ms = (
    heartbeat_seg_per_heartbeat_ms + c_point_per_heartbeat_ms + b_point_algos_per_heartbeat_ms
)

print(f"Heartbeat segmentation:          {heartbeat_seg_per_heartbeat_ms:.4f} ms/heartbeat")
print(f"C-point extraction:              {c_point_per_heartbeat_ms:.4f} ms/heartbeat")
print(f"12 B-point algorithms (summed):  {b_point_algos_per_heartbeat_ms:.4f} ms/heartbeat")
print(f"Total feature extraction:        {feature_extraction_time_per_heartbeat_ms:.4f} ms/heartbeat")


Heartbeat segmentation:          0.0678 ms/heartbeat
C-point extraction:              0.1349 ms/heartbeat
12 B-point algorithms (summed):  5.2175 ms/heartbeat
Total feature extraction:        5.4203 ms/heartbeat


### Load training data

We use the feature set of the best-performing ML pipeline reported in the paper (`MinMax_None_RFR_RR-Interval-Include-Nan`), i.e. the full feature set (12 base-algorithm outputs + estimated RR-interval) including samples where individual base algorithms failed to detect a B-point (`NaN`).

In [3]:
input_data_b_point = pd.read_csv(
    data_path.joinpath(f"rr-interval/{rater}/train_data_b_point_rr_interval_include_nan.csv"),
    index_col=[0, 1, 2, 3, 4, 5],
).astype(float)

X_b_point, y_b_point, groups_b_point, group_keys_b_point = bp.classification.utils.prepare_df_sklearn(
    data=input_data_b_point, label_col="b_point_sample_reference", subject_col="participant", print_summary=False
)

print(f"Number of samples: {X_b_point.shape[0]}")
print(f"Number of features: {X_b_point.shape[1]}")


Number of samples: 11239
Number of features: 13


### Build the reference ML pipeline

We rebuild the best-performing pipeline configuration found during hyperparameter search (`MinMaxScaler` + `RandomForestRegressor`, see `Analyse_Model_Parameters.ipynb` for the selected hyperparameters) rather than unpickling a `SklearnPipelinePermuter` result file, since computational cost measurements need a pipeline that is actually fit and predicted on this machine.

We split the data with the same `GroupKFold` (grouped by participant) scheme used during model selection so that the train/test sizes used for timing match one outer-CV fold of the original evaluation.

In [4]:
best_params = {
    "bootstrap": True,
    "criterion": "friedman_mse",
    "max_features": 0.8,
    "n_estimators": 250,
    "ccp_alpha": 0.001,
    "random_state": 0,
}

outer_cv = GroupKFold(n_splits=5)
train_idx, test_idx = next(outer_cv.split(X_b_point, y_b_point, groups=groups_b_point))

X_train, X_test = X_b_point[train_idx], X_b_point[test_idx]
y_train, y_test = y_b_point[train_idx], y_b_point[test_idx]

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")


Training set size: 8946 samples
Test set size: 2293 samples


### Training time

In [5]:
n_repeats_fit = 5
fit_times = []

for _ in range(n_repeats_fit):
    pipeline = Pipeline([
        ("scaler", MinMaxScaler()),
        ("clf", RandomForestRegressor(**best_params)),
    ])
    with measure_time() as timing_results:
        pipeline.fit(X_train, y_train)
    fit_times.append(timing_results["runtime_s"])

fit_times = np.array(fit_times)
print(f"Training time: {fit_times.mean():.3f} s ± {fit_times.std():.3f} s (n={n_repeats_fit} repeats, {X_train.shape[0]} training samples)")


Training time: 13.460 s ± 0.044 s (n=5 repeats, 8946 training samples)


### Batch inference time

Throughput of the fitted pipeline when predicting on the entire held-out fold at once (vectorized inference).

In [6]:
n_repeats_batch = 20
batch_times = []

for _ in range(n_repeats_batch):
    with measure_time() as timing_results:
        pipeline.predict(X_test)
    batch_times.append(timing_results["runtime_s"])

batch_times = np.array(batch_times)
time_per_sample_batch_ms = (batch_times / X_test.shape[0]) * 1000

print(f"Batch inference time ({X_test.shape[0]} samples): {batch_times.mean() * 1000:.3f} ms ± {batch_times.std() * 1000:.3f} ms")
print(f"Time per sample (batched): {time_per_sample_batch_ms.mean():.4f} ms ± {time_per_sample_batch_ms.std():.4f} ms")
print(f"Throughput: {X_test.shape[0] / batch_times.mean():.1f} samples/s")


Batch inference time (2293 samples): 62.159 ms ± 1.395 ms
Time per sample (batched): 0.0271 ms ± 0.0006 ms
Throughput: 36889.3 samples/s


### Single-sample inference latency

Realistic real-time / per-heartbeat use case: the model receives one feature vector (one heartbeat) at a time and has to return a B-point prediction immediately.

In [7]:
n_samples_single = min(200, X_test.shape[0])
rng = np.random.default_rng(0)
sample_indices = rng.choice(X_test.shape[0], size=n_samples_single, replace=False)

single_sample_times = []
for idx in sample_indices:
    x = X_test[idx : idx + 1]
    with measure_time() as timing_results:
        pipeline.predict(x)
    single_sample_times.append(timing_results["runtime_s"])

single_sample_times_ms = np.array(single_sample_times) * 1000

print(f"Single-sample latency (n={n_samples_single}):")
print(f"  mean:   {single_sample_times_ms.mean():.4f} ms")
print(f"  std:    {single_sample_times_ms.std():.4f} ms")
print(f"  median: {np.median(single_sample_times_ms):.4f} ms")
print(f"  p95:    {np.percentile(single_sample_times_ms, 95):.4f} ms")


Single-sample latency (n=200):
  mean:   6.0916 ms
  std:    3.1994 ms
  median: 5.7183 ms
  p95:    6.7333 ms


### Model size and complexity

In [8]:
rf = pipeline.named_steps["clf"]

buffer = BytesIO()
pickle.dump(pipeline, buffer)
model_size_kb = len(buffer.getvalue()) / 1024

node_counts = [estimator.tree_.node_count for estimator in rf.estimators_]
depths = [estimator.tree_.max_depth for estimator in rf.estimators_]

print(f"Serialized model size: {model_size_kb:.1f} KB")
print(f"Number of trees: {rf.n_estimators}")
print(f"Avg. nodes per tree: {np.mean(node_counts):.1f} ± {np.std(node_counts):.1f}")
print(f"Total nodes (all trees): {np.sum(node_counts)}")
print(f"Avg. tree depth: {np.mean(depths):.1f} ± {np.std(depths):.1f}")


Serialized model size: 92411.4 KB
Number of trees: 250
Avg. nodes per tree: 5253.5 ± 68.4
Total nodes (all trees): 1313382
Avg. tree depth: 25.3 ± 1.8


### Summary

In [ ]:
computational_cost_summary = pd.Series({
    "n_training_samples": X_train.shape[0],
    "n_test_samples": X_test.shape[0],
    "n_features": X_b_point.shape[1],
    "training_time_s_mean": fit_times.mean(),
    "training_time_s_std": fit_times.std(),
    "batch_inference_time_per_sample_ms_mean": time_per_sample_batch_ms.mean(),
    "batch_inference_time_per_sample_ms_std": time_per_sample_batch_ms.std(),
    "batch_throughput_samples_per_s": X_test.shape[0] / batch_times.mean(),
    "single_sample_latency_ms_mean": single_sample_times_ms.mean(),
    "single_sample_latency_ms_std": single_sample_times_ms.std(),
    "single_sample_latency_ms_median": np.median(single_sample_times_ms),
    "single_sample_latency_ms_p95": np.percentile(single_sample_times_ms, 95),
    "model_size_kb": model_size_kb,
    "n_trees": rf.n_estimators,
    "avg_nodes_per_tree": np.mean(node_counts),
    "avg_tree_depth": np.mean(depths),
    "feature_extraction_time_per_heartbeat_ms": feature_extraction_time_per_heartbeat_ms,
    "end_to_end_latency_ms_mean": feature_extraction_time_per_heartbeat_ms + single_sample_times_ms.mean(),
}, name="MinMax_None_RFR_RR-Interval-Include-Nan")

computational_cost_summary.to_frame()


### Save results

In [ ]:
#computational_cost_summary.to_frame().T.to_csv(
#    data_path.joinpath("computational_cost_ml_b_point.csv")
#)
